# TATN: Panasonic 25 C to CMU eVTOL

This notebook runs the same CMU core-seven protocol twice: (1) a TATN source trained once from the paper's Panasonic 25 C procedure, and (2) the authors' published `pre-25.pt`. CMU SOC references are used only after adaptation for evaluation.

In [9]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import battery_benchmark
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Package:', battery_benchmark.__file__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository ready: f5ee9a8a655b6815f804cb5d3b0a37267e2050ab
Package: /content/battery-soc-transfer-benchmark/src/battery_benchmark/__init__.py


## Shared protocol

- Panasonic source: all nine 25 C profiles, sampled at 1 Hz, 90-step non-overlapping windows.
- Self-trained source: paper-layout tensors; Adam, lr 1e-3, batch 64, max 300 epochs with label-safe early stopping.
- Released source: exact `pre-25.pt` and its required released-repository tensor layout.
- CMU adaptation: the same chronological complete-mission prefix used by DARM, reaching at least 8,778 seconds.
- Each CMU cell starts from the unchanged source checkpoint. No adapted state crosses cells.
- CMU target normalization uses only the Panasonic 25 C source ranges.

In [10]:
import pandas as pd
import torch
from IPython.display import display
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
DRIVE_OUTPUT = Path(PATHS['output_root'])
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv'
OFFICIAL_PRE25 = DRIVE_OUTPUT / 'reproductions/shen_2023/tatn_layout_ablation/official_pre25/pre-25.pt'
RESULT_ROOT = DRIVE_OUTPUT / 'benchmarks/panasonic_to_cmu_evtol/tatn'
CELLS = ['VAH01', 'VAH17', 'VAH27', 'VAH30', 'VAH15', 'VAH12', 'VAH11']
SOURCE_EPOCHS, TRANSFER_EPOCHS, BATCH_SIZE = 300, 150, 64
for path in (PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST, OFFICIAL_PRE25):
    assert path.exists(), f'Missing required input: {path}'

def run_variant(source_kind, output_name, released_checkpoint=None, scenario='unsupervised', trained_checkpoint=None):
    command = [
        sys.executable, str(REPO / 'scripts/benchmarks/run_tatn_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT / output_name),
        '--source-kind', source_kind, '--source-epochs', str(SOURCE_EPOCHS),
        '--transfer-epochs', str(TRANSFER_EPOCHS), '--batch-size', str(BATCH_SIZE),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu', '--resume',
        '--cells', *CELLS,
    ]
    if released_checkpoint is not None:
        command += ['--released-checkpoint', str(released_checkpoint)]
    if trained_checkpoint is not None:
        command += ['--trained-checkpoint', str(trained_checkpoint)]
    if scenario != 'unsupervised':
        command += ['--scenario', scenario]
    subprocess.run(command, check=True)
    result = pd.read_csv(RESULT_ROOT / output_name / 'condition_metrics.csv')
    display(result.pivot(index=['cell_id', 'condition'], columns='method', values=['overall_mae_percent', 'overall_rmse_percent']).round(3))
    return result

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('results:', RESULT_ROOT)

device: cuda
results: /content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/tatn


## Result 1 - source trained from the Panasonic 25 C paper protocol

In [11]:
trained_results = run_variant('trained', 'trained_paper_layout')

overall_mae_percent         overall_rmse_percent  \
method                               Source only    TATN          Source only   
cell_id condition                                                               
VAH01   baseline_protocol                  6.677  22.006                8.047   
VAH11   reduction_20_percent               4.391  28.676                9.486   
VAH12   short_400_s                        8.659   8.659               12.120   
VAH15   extended_1000_s                    7.262  36.182                8.054   
VAH17   baseline_protocol                  6.854   6.854                7.754   
VAH27   baseline_protocol                  6.074  12.970                6.564   
VAH30   chamber_35_c                       9.961   9.961               13.058   

                                      
method                          TATN  
cell_id condition                     
VAH01   baseline_protocol     23.971  
VAH11   reduction_20_percent  30.264  
VAH12   short_400_s           12.120  
VAH15   extended_1000_s       37.376  
VAH17   baseline_protocol      7.754  
VAH27   baseline_protocol     15.731  
VAH30   chamber_35_c          13.058

## Result 2 - authors' published Panasonic 25 C checkpoint

In [12]:
released_results = run_variant('released', 'released_pre25', OFFICIAL_PRE25)

overall_mae_percent         overall_rmse_percent  \
method                               Source only    TATN          Source only   
cell_id condition                                                               
VAH01   baseline_protocol                 26.694  26.694               30.213   
VAH11   reduction_20_percent              24.223  23.341               34.929   
VAH12   short_400_s                       20.296  20.296               22.830   
VAH15   extended_1000_s                   28.866  27.213               32.977   
VAH17   baseline_protocol                 27.466  26.394               30.938   
VAH27   baseline_protocol                 25.009  22.106               29.058   
VAH30   chamber_35_c                      26.992  26.907               30.237   

                                      
method                          TATN  
cell_id condition                     
VAH01   baseline_protocol     30.213  
VAH11   reduction_20_percent  33.635  
VAH12   short_400_s           22.830  
VAH15   extended_1000_s       31.511  
VAH17   baseline_protocol     30.210  
VAH27   baseline_protocol     26.265  
VAH30   chamber_35_c          30.000

## Semisupervised protocol

For each CMU cell, the earliest adaptation mission contributes the first 50% of its SOC reference labels; all other adaptation missions remain unlabelled and use selected pseudolabels. Test missions remain held out. These CMU SOC references are development estimates rather than laboratory ground truth. The two runs below reuse the source checkpoints from Results 1 and 2 and write to separate output directories.

## Result 3 - semisupervised, source trained from the Panasonic 25 C paper protocol

In [13]:
TRAINED_SOURCE_CHECKPOINT = RESULT_ROOT / 'trained_paper_layout/source_25C.pt'
assert TRAINED_SOURCE_CHECKPOINT.exists(), 'Run Result 1 first to create the trained source checkpoint'
trained_semisupervised_results = run_variant(
    'trained', 'trained_paper_layout_semisupervised',
    scenario='semisupervised', trained_checkpoint=TRAINED_SOURCE_CHECKPOINT,
)

overall_mae_percent         overall_rmse_percent  \
method                               Source only    TATN          Source only   
cell_id condition                                                               
VAH01   baseline_protocol                  6.677  17.076                8.048   
VAH11   reduction_20_percent               4.392  22.081                9.487   
VAH12   short_400_s                        8.660  10.988               12.120   
VAH15   extended_1000_s                    7.262  14.695                8.053   
VAH17   baseline_protocol                  6.854  15.222                7.754   
VAH27   baseline_protocol                  6.074  15.746                6.564   
VAH30   chamber_35_c                       9.963  10.324               13.059   

                                      
method                          TATN  
cell_id condition                     
VAH01   baseline_protocol     19.408  
VAH11   reduction_20_percent  23.778  
VAH12   short_400_s           12.214  
VAH15   extended_1000_s       18.762  
VAH17   baseline_protocol     17.523  
VAH27   baseline_protocol     18.329  
VAH30   chamber_35_c          11.938

## Result 4 - semisupervised, authors' published Panasonic 25 C checkpoint

In [14]:
released_semisupervised_results = run_variant(
    'released', 'released_pre25_semisupervised', OFFICIAL_PRE25,
    scenario='semisupervised',
)

overall_mae_percent         overall_rmse_percent  \
method                               Source only    TATN          Source only   
cell_id condition                                                               
VAH01   baseline_protocol                 26.695  19.019               30.213   
VAH11   reduction_20_percent              24.218  14.385               34.926   
VAH12   short_400_s                       20.297  17.994               22.831   
VAH15   extended_1000_s                   28.866  25.088               32.977   
VAH17   baseline_protocol                 27.466  22.059               30.938   
VAH27   baseline_protocol                 25.008  17.278               29.058   
VAH30   chamber_35_c                      26.992  25.837               30.237   

                                      
method                          TATN  
cell_id condition                     
VAH01   baseline_protocol     22.472  
VAH11   reduction_20_percent  19.215  
VAH12   short_400_s           20.799  
VAH15   extended_1000_s       30.504  
VAH17   baseline_protocol     26.390  
VAH27   baseline_protocol     20.810  
VAH30   chamber_35_c          29.107